In [1]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import traceback
from typing import Dict, List, Any, Tuple

import pandas as pd
from tqdm.auto import tqdm
import textgrid

In [37]:
# sofa configs
SOFA_REPO = Path("/home/hbli/songformer/repo/SOFA/")   # <-- 改成你自己的
SOFA_CKPT = Path("/home/hbli/songformer/repo/SOFA/ckpt/tgm_en_v100.ckpt")
SOFA_DICTIONARY = Path("/home/hbli/songformer/repo/SOFA/dictionary/tgm_sofa_dict.txt")

# lyrics json
JSONL_PATH = Path("/mnt/ssd/hbli/datasets/harmonixset/harmonixset_lrclib_results.jsonl")

# audio
AUDIO_DIR = Path("/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios")

OUTPUT_DIR = Path("/mnt/ssd/hbli/songformer/runs/sofa_eda")

# SOFA temp dir
# wav and lab should be placed in the same dir for SOFA to work
SEGMENTS_ROOT = OUTPUT_DIR / "segments"
SEGMENT_SUBDIR = SEGMENTS_ROOT / "harmonixset"

# SOFA mode
# force: 强制整段文本都参与对齐
# match: 允许只匹配文本中的最可能连续子段（对 excerpt 更友好）
SOFA_MODE = "match"

# output formats
SOFA_OUT_FORMATS = "textgrid,trans"

# if output confidence scores
SAVE_CONFIDENCE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_SUBDIR.mkdir(parents=True, exist_ok=True)

print("SOFA_REPO:", SOFA_REPO)
print("SOFA_CKPT:", SOFA_CKPT)
print("SOFA_DICTIONARY:", SOFA_DICTIONARY)
print("JSONL_PATH:", JSONL_PATH)
print("AUDIO_DIR:", AUDIO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

SOFA_REPO: /home/hbli/songformer/repo/SOFA
SOFA_CKPT: /home/hbli/songformer/repo/SOFA/ckpt/tgm_en_v100.ckpt
SOFA_DICTIONARY: /home/hbli/songformer/repo/SOFA/dictionary/tgm_sofa_dict.txt
JSONL_PATH: /mnt/ssd/hbli/datasets/harmonixset/harmonixset_lrclib_results.jsonl
AUDIO_DIR: /mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios
OUTPUT_DIR: /mnt/ssd/hbli/songformer/runs/sofa_eda


In [38]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def filter_usable(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    usable = []
    for r in records:
        if r.get("match_score", 0) > MIN_MATCH_SCORE and r.get("has_plain") is REQUIRE_HAS_PLAIN:
            usable.append(r)
    return usable

def resolve_audio_path(file_id: str, audio_dir: Path) -> Path:
    candidates = [
        audio_dir / f"{file_id}.wav",
        audio_dir / f"HX_{file_id}.wav",
    ]
    for p in candidates:
        if p.exists():
            return p

    # fallback: contains match
    contains = sorted(audio_dir.glob(f"*{file_id}*.wav"))
    if contains:
        return contains[0]

    raise FileNotFoundError(f"Cannot find wav for {file_id}")

In [39]:
TEST_FILE_IDS = [
    "0008_america",
    "0003_6foot7foot",
]

# filter condition
MIN_MATCH_SCORE = 0.5
REQUIRE_HAS_PLAIN = True

all_records = load_jsonl(JSONL_PATH)
usable_records = filter_usable(all_records)
usable_map = {r["File"]: r for r in usable_records}

In [40]:
selected_records = []
for file_id in TEST_FILE_IDS:
    if file_id in usable_map:
        selected_records.append(usable_map[file_id])
    else:
        print(f"WARNING: {file_id} not found or not usable.")

preview_df = pd.DataFrame([
    {
        "File": r["File"],
        "Title": r.get("Title"),
        "Artist": r.get("Artist"),
        "match_score": r.get("match_score"),
        "has_plain": r.get("has_plain"),
        "has_synced": r.get("has_synced"),
    }
    for r in selected_records
])

print("Total records:", len(all_records))
print("Usable records:", len(usable_records))
preview_df

Total records: 912
Usable records: 869


,File,Title,Artist,match_score,has_plain,has_synced
0,0008_america,America,Spın̈al Tap,0.991057,True,False
1,0003_6foot7foot,6 Foot 7 Foot,Lil Wayne,0.950000,True,True


In [41]:
def normalize_text_for_lab(text: str) -> str:
    """
    把 plain lyrics 清洗成 SOFA .lab 需要的一行空格分词格式。
    """
    text = text.replace("’", "'").replace("‘", "'").replace("—", " ").replace("–", " ").replace("-", " ")
    text = text.lower()

    # 去掉大部分标点，保留字母、数字、空格、撇号
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_line_to_clean_words(raw_line: str) -> List[str]:
    norm = normalize_text_for_lab(raw_line)
    if not norm:
        return []
    return norm.split()

def parse_plain_lyrics_structure(plain_lyrics: str, valid_dict_words: set) -> List[Dict[str, Any]]:
    """
    保留原始 line 结构，同时只保留 dictionary 中存在的 clean words。
    这样 line 聚合时能和 SOFA 实际输入对齐。
    """
    entries = []
    for raw_line in plain_lyrics.split("\n"):
        clean_words = split_line_to_clean_words(raw_line)
        clean_words = [w for w in clean_words if w in valid_dict_words]

        entries.append(
            {
                "raw_line": raw_line,
                "is_blank": len(raw_line.strip()) == 0,
                "clean_words": clean_words,
                "num_words": len(clean_words),
            }
        )
    return entries

def format_lrc_time(seconds: float) -> str:
    if seconds is None:
        return "00:00.00"
    cs = int(round(seconds * 100))
    mm = cs // 6000
    ss = (cs % 6000) // 100
    cc = cs % 100
    return f"{mm:02d}:{ss:02d}.{cc:02d}"

In [42]:
def load_dictionary_words(dict_path: Path) -> set:
    words = set()
    with open(dict_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            word = line.split("\t")[0].strip().lower()
            words.add(word)
    return words

DICT_WORDS = load_dictionary_words(SOFA_DICTIONARY)

def inspect_oov(record: Dict[str, Any], valid_dict_words: set) -> Dict[str, Any]:
    norm_all = normalize_text_for_lab(record["plain_lyrics"])
    words = norm_all.split() if norm_all else []
    unique_words = sorted(set(words))
    oov = [w for w in unique_words if w not in valid_dict_words]
    return {
        "File": record["File"],
        "Title": record.get("Title"),
        "Artist": record.get("Artist"),
        "num_unique_words": len(unique_words),
        "num_oov_words": len(oov),
        "oov_ratio": 0.0 if len(unique_words) == 0 else len(oov) / len(unique_words),
        "first_30_oov": oov[:30],
    }

oov_df = pd.DataFrame([inspect_oov(r, DICT_WORDS) for r in selected_records])
oov_df

,File,Title,Artist,num_unique_words,num_oov_words,oov_ratio,first_30_oov
0,0008_america,America,Spın̈al Tap,84,5,0.059524,"[an', byes, pta, reachin', womens]"
1,0003_6foot7foot,6 Foot 7 Foot,Lil Wayne,386,32,0.082902,"['f', 'less, backin', bodybuilder, buyin', bx,..."


In [43]:
def safe_symlink_or_copy(src: Path, dst: Path):
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def prepare_sofa_inputs(record: Dict[str, Any], segment_subdir: Path, valid_dict_words: set) -> Dict[str, Any]:
    audio_path = resolve_audio_path(record["File"], AUDIO_DIR)
    audio_stem = audio_path.stem

    lab_path = segment_subdir / f"{audio_stem}.lab"
    wav_path = segment_subdir / f"{audio_stem}.wav"

    line_entries = parse_plain_lyrics_structure(record["plain_lyrics"], valid_dict_words)

    flat_words = []
    for e in line_entries:
        flat_words.extend(e["clean_words"])

    lab_text = " ".join(flat_words).strip()

    # wav 放到 SOFA 输入目录
    safe_symlink_or_copy(audio_path, wav_path)

    # 写 lab
    with open(lab_path, "w", encoding="utf-8") as f:
        f.write(lab_text)

    meta = {
        "audio_path": str(audio_path),
        "audio_stem": audio_stem,
        "sofa_wav_path": str(wav_path),
        "sofa_lab_path": str(lab_path),
        "flat_words": flat_words,
        "line_entries": line_entries,
        "num_flat_words": len(flat_words),
    }
    return meta

prepared = {}
for r in selected_records:
    prepared[r["File"]] = prepare_sofa_inputs(r, SEGMENT_SUBDIR, DICT_WORDS)

pd.DataFrame([
    {
        "File": k,
        "audio_stem": v["audio_stem"],
        "num_flat_words": v["num_flat_words"],
        "lab_preview": " ".join(v["flat_words"][:20]),
    }
    for k, v in prepared.items()
])

,File,audio_stem,num_flat_words,lab_preview
0,0008_america,HX_0008_america,111,we came like babies from our home across the s...
1,0003_6foot7foot,HX_0003_6foot7foot,798,six foot seven foot eight foot bunch six foot ...


In [44]:
def run_sofa_inference():
    cmd = [
        sys.executable,
        str(SOFA_REPO / "infer.py"),
        "--ckpt", str(SOFA_CKPT),
        "--folder", str(SEGMENTS_ROOT),
        "--g2p", "Dictionary",
        "--dictionary", str(SOFA_DICTIONARY),
        "--in_format", "lab",
        "--out_formats", SOFA_OUT_FORMATS,
        "--mode", SOFA_MODE,
    ]

    if SAVE_CONFIDENCE:
        cmd.append("--save_confidence")

    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=str(SOFA_REPO),
        capture_output=True,
        text=True
    )

    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"SOFA inference failed with code {result.returncode}")


import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

run_sofa_inference()

Running command:
/home/hbli/songformer/env/miniforge3/envs/SOFA/bin/python /home/hbli/songformer/repo/SOFA/infer.py --ckpt /home/hbli/songformer/repo/SOFA/ckpt/tgm_en_v100.ckpt --folder /mnt/ssd/hbli/songformer/runs/sofa_eda/segments --g2p Dictionary --dictionary /home/hbli/songformer/repo/SOFA/dictionary/tgm_sofa_dict.txt --in_format lab --out_formats textgrid,trans --mode match --save_confidence
STDOUT:
 'torchaudio' installed and imported.
Loaded 2 samples.

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|██████████| 2/2 [00:08<00:00,  0.23it/s]
Post-processing...
Saving TextGrids...
Saving transcriptions.csv...
saving confidence...
Output files are saved to the same folder as the input wav files.

STDERR:
 /home/hbli/songformer/env/miniforge3/envs/SOFA/lib/python3.11/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resourc

In [45]:
IGNORE_MARKS = {"", "SP", "AP", "<SP>", "<AP>", "pau", "cl"}

def get_textgrid_path(audio_stem: str) -> Path:
    return SEGMENT_SUBDIR / "TextGrid" / f"{audio_stem}.TextGrid"

def get_confidence_csv_path() -> Path:
    return SEGMENT_SUBDIR / "confidence" / "confidence.csv"

def parse_textgrid_word_phone(textgrid_path: Path) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    tg = textgrid.TextGrid.fromFile(str(textgrid_path))

    word_tier = None
    phone_tier = None
    for tier in tg.tiers:
        if tier.name == "words":
            word_tier = tier
        elif tier.name == "phones":
            phone_tier = tier

    if word_tier is None or phone_tier is None:
        raise ValueError(f"Expected 'words' and 'phones' tiers in {textgrid_path}")

    word_items = []
    for itv in word_tier:
        mark = (itv.mark or "").strip()
        if mark in IGNORE_MARKS:
            continue
        word_items.append(
            {
                "word": mark,
                "start_time": float(itv.minTime),
                "end_time": float(itv.maxTime),
            }
        )

    phone_items = []
    for itv in phone_tier:
        mark = (itv.mark or "").strip()
        if mark in IGNORE_MARKS:
            continue
        phone_items.append(
            {
                "phone": mark,
                "start_time": float(itv.minTime),
                "end_time": float(itv.maxTime),
            }
        )

    return word_items, phone_items

def load_confidence_map(conf_path: Path) -> Dict[str, Any]:
    if not conf_path.exists():
        return {}
    df = pd.read_csv(conf_path)
    return dict(zip(df["name"].astype(str), df["confidence"]))

confidence_map = load_confidence_map(get_confidence_csv_path())
confidence_map

{'HX_0008_america': 0.3404539824988619,
 'HX_0003_6foot7foot': 0.3311459978706246}

In [46]:
def build_aligned_outputs(record: Dict[str, Any], meta: Dict[str, Any], confidence_map: Dict[str, Any]) -> Dict[str, Any]:
    audio_stem = meta["audio_stem"]
    tg_path = get_textgrid_path(audio_stem)

    if not tg_path.exists():
        raise FileNotFoundError(f"Missing TextGrid: {tg_path}")

    word_items, phone_items = parse_textgrid_word_phone(tg_path)

    # 默认 force 模式下，这里应与 flat_words 长度一致
    flat_words = meta["flat_words"]
    if len(word_items) != len(flat_words):
        print(f"WARNING: word count mismatch for {record['File']}: "
              f"SOFA={len(word_items)} vs input={len(flat_words)}")

    # word_aligned_lyrics：每个词一行
    word_aligned_lyrics = "\n".join(
        [f"[{format_lrc_time(x['start_time'])}] {x['word']}" for x in word_items]
    )

    # line-level：按原始 \n 结构聚合
    line_items = []
    cursor = 0
    for line_idx, e in enumerate(meta["line_entries"]):
        if e["is_blank"]:
            line_items.append(
                {
                    "line_idx": line_idx,
                    "raw_line": "",
                    "start_time": None,
                    "end_time": None,
                    "num_words": 0,
                }
            )
            continue

        n = e["num_words"]
        chunk = word_items[cursor: cursor + n]
        cursor += n

        if len(chunk) == 0:
            line_items.append(
                {
                    "line_idx": line_idx,
                    "raw_line": e["raw_line"],
                    "start_time": None,
                    "end_time": None,
                    "num_words": 0,
                }
            )
        else:
            line_items.append(
                {
                    "line_idx": line_idx,
                    "raw_line": e["raw_line"],
                    "start_time": float(chunk[0]["start_time"]),
                    "end_time": float(chunk[-1]["end_time"]),
                    "num_words": len(chunk),
                }
            )

    # line_aligned_lyrics：保留 stanza 空行
    line_lrc_lines = []
    for x in line_items:
        if x["raw_line"] == "":
            line_lrc_lines.append("")
        elif x["start_time"] is None:
            line_lrc_lines.append(x["raw_line"])
        else:
            line_lrc_lines.append(f"[{format_lrc_time(x['start_time'])}] {x['raw_line']}")

    line_aligned_lyrics = "\n".join(line_lrc_lines)

    out = dict(record)
    out["audio_path"] = meta["audio_path"]
    out["sofa_mode"] = SOFA_MODE
    out["sofa_model_ckpt"] = str(SOFA_CKPT)
    out["sofa_dictionary"] = str(SOFA_DICTIONARY)
    out["sofa_confidence"] = confidence_map.get(audio_stem, None)

    out["word_alignment_items"] = word_items
    out["phone_alignment_items"] = phone_items
    out["line_alignment_items"] = line_items

    out["word_aligned_lyrics"] = word_aligned_lyrics
    out["line_aligned_lyrics"] = line_aligned_lyrics

    return out

In [47]:
results = []
status_rows = []

for r in tqdm(selected_records, desc="Build SOFA aligned outputs"):
    try:
        out = build_aligned_outputs(r, prepared[r["File"]], confidence_map)
        results.append(out)

        audio_stem = prepared[r["File"]]["audio_stem"]
        save_path = OUTPUT_DIR / f"{audio_stem}.json"

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

        status_rows.append(
            {
                "File": r["File"],
                "audio_stem": audio_stem,
                "status": "ok",
                "num_words": len(out["word_alignment_items"]),
                "num_lines": len(out["line_alignment_items"]),
                "confidence": out["sofa_confidence"],
                "output_json": str(save_path),
            }
        )
    except Exception as e:
        status_rows.append(
            {
                "File": r["File"],
                "status": "error",
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
        )

status_df = pd.DataFrame(status_rows)
status_df

Build SOFA aligned outputs:   0%|          | 0/2 [00:00<?, ?it/s]

,File,audio_stem,status,num_words,num_lines,confidence,output_json
0,0008_america,HX_0008_america,ok,1,27,0.340454,/mnt/ssd/hbli/songformer/runs/sofa_eda/HX_0008...
1,0003_6foot7foot,HX_0003_6foot7foot,ok,34,114,0.331146,/mnt/ssd/hbli/songformer/runs/sofa_eda/HX_0003...


In [54]:
import glob
glob.glob("/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet/*yours.wav")

['/mnt/ssd/hbli/datasets/songformer/songformbench/data/audios/HarmonixSet/BHX_0748_imyours.wav']